# Day 3 — OOP for Pipelines: Composition Over Inheritance

## Objective
Demonstrate modular Object-Oriented Design for analytics pipelines. Use Abstract Base Classes (`abc.ABC`), interchangeable concrete `Step` classes, `Pipeline` composition, runtime step swapping, Python encapsulation conventions, and guidelines on choosing between functions and classes.

## 1. Abstract Base Class Interface (`Step`)

The `Step` class defines a strict contract using `ABC` and `@abstractmethod`. Directly instantiating `Step` raises a `TypeError`.

In [1]:
import sys
from pathlib import Path
repo_root = Path.cwd().resolve()
sys.path.insert(0, str(repo_root / 'src'))

from task_analytics import (
    Step, CleanDataStep, NormalizeDataStep, FilterDataStep, PriorityFilterStep, Pipeline, EncapsulationDemo, clean_text
)

try:
    step = Step()
except TypeError as e:
    print('Caught expected TypeError when instantiating Step abstract class:')
    print('  -->', e)

Caught expected TypeError when instantiating Step abstract class:
  --> Can't instantiate abstract class Step with abstract method execute


## 2. Concrete Pipeline Steps & Composition

Build an initial pipeline containing `CleanDataStep`, `NormalizeDataStep`, and `FilterDataStep`.

In [2]:
raw_tasks = [
    {'title': '  Fix Authentication Bug ', 'status': ' IN_PROGRESS ', 'priority': ' HIGH '},
    None,
    {},
    {'title': ' Write Unit Tests ', 'status': ' COMPLETED ', 'priority': ' LOW '},
    {'title': ' Refactor Pipeline Code ', 'status': ' IN_PROGRESS ', 'priority': ' HIGH '},
]

print(f'Raw tasks count: {len(raw_tasks)}')
pipeline = Pipeline([
    CleanDataStep(required_keys=['title']),
    NormalizeDataStep(target_fields=['status', 'priority']),
    FilterDataStep(field='status', value='in_progress'),
])

result_1 = pipeline.run(raw_tasks)
print(f'Filtered results count (status==in_progress): {len(result_1)}')
for item in result_1:
    print('  ->', item)

Raw tasks count: 5
Filtered results count (status==in_progress): 2
  -> {'title': '  Fix Authentication Bug ', 'status': 'in_progress', 'priority': 'high'}
  -> {'title': ' Refactor Pipeline Code ', 'status': 'in_progress', 'priority': 'high'}


## 3. Runtime Step Swapping (Open/Closed Principle)

Swap the status filter step with `PriorityFilterStep(priority='high')` at runtime without modifying the `Pipeline` class.

In [3]:
print('Swapping step at index 2 to PriorityFilterStep(priority=high)...')
pipeline.replace_step(2, PriorityFilterStep(priority='high'))
result_2 = pipeline.run(raw_tasks)
print(f'Updated pipeline output (priority==high): {len(result_2)} items')
for item in result_2:
    print('  ->', item)

Swapping step at index 2 to PriorityFilterStep(priority=high)...
Updated pipeline output (priority==high): 2 items
  -> {'title': '  Fix Authentication Bug ', 'status': 'in_progress', 'priority': 'high'}
  -> {'title': ' Refactor Pipeline Code ', 'status': 'in_progress', 'priority': 'high'}


## 4. Encapsulation Conventions in Python

Demonstrate Python encapsulation: Public, Protected (`_`), and Private (`__` name mangling).

In [4]:
demo = EncapsulationDemo(name='Pipeline1', protected_val='ConfigVal', private_val='SecretKey')
print('Public attribute:', demo.name)
print('Protected attribute (_protected_val):', demo._protected_val)
try:
    _ = demo.__private_val
except AttributeError as e:
    print('Caught expected AttributeError on direct private access:', e)
print('Access via name mangling (_EncapsulationDemo__private_val):', getattr(demo, '_EncapsulationDemo__private_val'))

Public attribute: Pipeline1
Protected attribute (_protected_val): ConfigVal
Caught expected AttributeError on direct private access: 'EncapsulationDemo' object has no attribute '__private_val'
Access via name mangling (_EncapsulationDemo__private_val): SecretKey


## 5. Function vs Class Guidelines

- **Use Pure Functions**: When performing stateless data transformations (e.g. `clean_text`).
- **Use Classes**: When encapsulating internal state, configuration, or polymorphic interfaces (e.g. `Step`).

In [5]:
raw_str = '   Sample Task Title STRING   '
cleaned_str = clean_text(raw_str)
print(f"Pure function clean_text('{raw_str}') -> '{cleaned_str}'")

Pure function clean_text('   Sample Task Title STRING   ') -> 'sample task title string'


## Conclusion & Key Takeaways

- **Composition Over Inheritance**: `Pipeline` contains `Step` instances, allowing dynamic workflow assembly.
- **Extensibility**: Steps can be added or swapped dynamically without modifying the host `Pipeline` class.